# Integrated Intelligent Systems: Tutorial 3 - Body Motion Problem (Scientific Object) (3 Pts)

This notebook introduces a formulation body motion problem, central to designing and implementing Cognition-Enabled Robot Agents, as a description logic system.

## Part 1: Robot-World Manipulation System as Scientific Object (E1–E6) (1.5 Pt)

**Title:** Formalizing the Robot Agent: From Architecture to Description Logic Reasoners

![Alternative Text](RWMS.png)

### Scenario Background
An autonomous robotic agent (the PR2 in the EASE apartment) is deployed to execute a manipulation task ("Serve Eva a drink"). To validate this agent scientifically, we must formalize the system as a **Robot-World Manipulation System (RWMS)**, separate the controlling subsystem from the controlled body-world system, define trajectories as trees, and assess runs against an external problem standard using a description logic reasoner.

---

### Multi-Question Student Task Sheet (0.25 Pt x 6)

1. **E1 · Architectural Cut & Subsystem Separation**
   - In the graphical representation of the RWMS, a strict architectural "cut" is placed between the manipulation executive and the body-world system ($[B \bowtie W]$). Explain the nature of this cut: why is the manipulation executive classified as a `dul:Agent` (non-physical program) while the body-world system is classified under physical objects, and how do they interact exclusively across the boundary?

2. **E2 · Forward-Branching Time & World Choices**
   - During an operation on a fridge door or container seal, the same action token (e.g., a position-controlled pull) can lead to completely divergent physical outcomes depending on world choices (e.g., seal releases vs. seal holds tight). Draw or describe the structure of the resulting courses, and explain why the possible courses form a tree that branches *only forward* from the initial state $x_0$.

3. **E3 · Trajectory Classification & Priority Hierarchies**
   - The problem standard $P$ sorts every physical trajectory into one of four classes: **SOL** (Solution), **REPD** (Repaired), **NON** (Nonachievement), and **PROH** (Prohibited). Consider two runs that both successfully place the drink within Eva's reach, but where Run A tears off the container handle (`P4` violation) while Run B maintains licensed contact. Classify both runs and explain why the classifier grades the *entire course* rather than just the final end-state.

4. **E4 · Description Logic Partonomy & DUL Constraints**
   - When formalizing the RWMS in description logic under DOLCE Ultra Lite (DUL), we define an RWMS as having *exactly one* executive and *exactly one* body-world system (`RWMS = (=1 hasComponent.ManipExecutive) ⊓ (=1 hasComponent.BodyWorldSystem)`). Using the transitivity of `dul:isPartOf`, explain how a reasoner (such as HermiT) automatically entails that the PR2's head camera is a component of the RWMS without explicitly hardcoding every part-whole link.

5. **E5 · Ontological Inconsistency & Entailment Limits**
   - An inconsistency in description logic serves as a strict analytical tool. Formulate a competency question that results in an ontological inconsistency when attempting to classify the *manipulation problem itself* as a physical component inside the RWMS. Furthermore, explain why numerical physical laws (such as Coulomb friction $2 \cdot \mu \cdot F \ge m \cdot (g + a)$) must be handled outside description logic by the semantic digital twin's causal model.

6. **E6 · Strength vs. Truth in Agent Validation**
   - The design thesis dictates that an agent decides within its believed causality, which can be tested for logical *strength* via competency queries and for *truth* via physical experiments. Compare how an axiom set that lacks a friction law ($\text{K}_0$) performs on competency queries versus an axiomatization containing explicit dynamical laws ($\text{K}_1/\text{K}_2$), and state what kind of test is required to validate the actual physical truth of those beliefs.

---------------------

#### E1

The way I understand the "cut" is that even though the executive runs inside the PR2.. we still separate the program deciding what to do from the physical body carrying it out. The executive is a `dul:Agent` because it reads observations, decides what to do next and commits to an action.. the body–world system \([B \bowtie W]\) is physical because it includes the PR2’s body and the things it actually interacts with.. like its gripper pulling on the fridge handle

So for ex. if the executive decides to open the fridge.. what it sends across the cut is like an "action token" like “pull the handle”.. the gripper then pulls, BUT the seal might still hold.. so the executive needs an "observation" back from the body–world system to tell whether the door actually opened. Those are the only two things crossing the cut.. action tokens going out and observations coming back



#### E2

Starting from $x_0$.. the executive sends the same action token in both cases, “pull the handle”.. then the course splits depending on how the seal responds. If the seal releases, the door swings open and the bottle is reachable.. BUT if the seal holds tight, the position controller keeps pulling and the handle can tear off. That split is the "world choice".. the executive chose the pull and each possible response gives us a different branch

The way I understand "only forward" is that both courses share the same history up to the pull.. from there we have two possible ways things can continue. Each path from $x_0$ is one possible course with one specific past.. even if the PR2 ends up in the same position later on both branches.. what happened to get there was different, so the two histories stay separate in the tree



#### E3

I’d classify "Run A as PROH".. the drink does end up within Eva’s reach.. BUTTT tearing off the handle violates `P4`.. and a hard prevention takes priority even if the robot still achieves the goal afterward. "Run B is SOL" assuming the other conditions also held.. the robot kept contact within what was allowed and placed the drink within Eva’s reach without needing a repair

If we only checked “is the drink within Eva’s reach?” at the end.. both runs would get a yes!! But the task also says what has to stay true along the way and what must never happen.. so once the handle is torn off in Run A.. delivering the drink afterward can’t undo that violation. That’s why $P$ grades the "whole course"



#### E4

The way I understand it is that we just give HermiT the links one level at a time.. the head camera is part of the PR2’s body, the body is a component of the body–world system and the body–world system is a component of the RWMS.. however in DUL being a component also means being part of that whole.. so the reasoner can follow the chain "" head camera >> PR2 body >> body–world system >> RWMS ""

Since `dul:isPartOf` is "transitive".. HermiT can infer `headCamera dul:isPartOf RWMS` from those links.. we don’t have to hardcode a separate camera-to-RWMS link. The wording matters a bit here.. the camera is an "indirect part" of the RWMS.. while the “exactly one” conditions count the executive and the body–world system through `hasComponent`



#### E5

My competency question would be "Can the manipulation problem ‘Serve Eva a drink’ itself be a physical component of the RWMS??".. if we add that claim and run HermiT, the ontology becomes inconsistent. The problem is a `dul:Description`.. it defines what counts as an acceptable run. The RWMS allows only a manipulation executive or a physical object as a component and DUL keeps a description disjoint from both. The relation that fits here is `assessedAgainst`.. the RWMS is assessed against the problem

For the friction example.. the ontology can say that the carton has a mass and a friction coefficient (based on the lec).. BUT figuring out whether the gripper can actually hold it during a lift needs a calculation. The twin’s causal model checks $2 \cdot \mu \cdot F \ge m \cdot (g + a)$.. basically, is there enough friction from the two gripper contacts to support the carton while it accelerates upward? That means multiplying and comparing values across several properties.. which the description logic reasoner can’t do on its own



#### E6

Say we ask “will the wet carton slip if the PR2 lifts it with this grip force?”.. $K_0$ might know the carton’s mass and friction coefficient, but it has no law connecting those values to what happens during the lift.. so the query stays unanswered. With $K_1$/$K_2$, the relevant dynamical laws are explicit.. given the grip force, friction, mass and acceleration, the model can predict whether the carton will hold or slip. That’s what "strength" means here.. can we actually derive the prediction from what the agent believes?

Now to check "truth".. I would actually have the PR2 do the lift with a real carton under the same conditions and measure whether it actually slips.. then repeat the test with different grip forces around the point where the model predicts it should start slipping. If the carton slips while the model predicts it should hold, then the beliefs were strong enough to give an answer.. BUT that answer didn’t match what happened in the real lift

------------------

## Part 2: Robot Motion Problem and Description Logics (1.5 Pt)

**Title:** Formulating and Querying the Robot Motion Problem with Knowledge Bases & Description Logics

![Alternative Text](PR2.png)

### Scenario Background
Using the interactive Jupyter notebook and RViz simulation environment (`bmp.rviz`, `setup_demo()`), the PR2 mobile manipulator interacts with the EASE apartment model (including the fridge and various kitchen drawers). Instead of hardcoding trajectories, the robotic executive queries a knowledge base (represented via Description Logics / Prolog reasoning) to link task requests to state changes, causes (motions), and capabilities.

[Go to the programming Lab](https://binder.intel4coro.de/v2/gh/maltehue/giskard-examples/bmp_vrb?urlpath=/lab/tree/notebooks/bodyMotionProblem.ipynb)
---

### Multi-Question Student Task Sheet (0.25 Pt x 6)

1. **E7 · Knowledge Base Grounding & Task Requests**
   - Execute the initial query `taskRequest(Request, StateChange).`Examine the resulting solution bindings (e.g., `{'Request': 'open_fridge', 'StateChange': 'open(fridge)'}` and various drawer operations). Explain how mapping abstract task requests to discrete state changes forms the top-level specification layer for the manipulation executive.

2. **E8 · Extending Queries for Causal Motions & Parameters**
   - Extend the query to incorporate motion causes: `taskRequest(Request, StateChange), causes(Motion, StateChange, MotionParam).` Explain how the reasoner bridges the gap between a desired symbolic state change and continuous motion parameters (such as `envJointGoal`, target joint angles, and handle identifiers).

3. **E9 · Integrating Robot Capabilities (`canPerform`)**
   - Finalize the query by adding the robot capability constraint: `taskRequest(Request, StateChange), causes(Motion, StateChange, MotionParam), canPerform(Robot, Motion, MotionParam).`Describe what role the `Robot` variable (`'pr2'`) plays in filtering feasible trajectories through the reasoner.

4. **E10 · Specializing Request Scope (`open_fridge` vs. `open_anything`)**
   - Modify the query request parameter from general categories like `open_anything` to specifically target `open_fridge`. Explain how refining the scope in the knowledge base reduces the solution search tree and prevents unauthorized or unintended state changes in the kitchen environment.

5. **E11 · Visualization and Simulation Linkage in RViz**
   - With the background shell command executing `rviz -d .../bmp.rviz` and displaying the PR2 interacting with the fridge handle, explain how the motion parameters derived from the reasoner's output (`[0.8, b'fridge_door_joint', b'fridge_door_handle']`) are executed as physical joint trajectories in the simulation.

6. **E12 · Ontological Limits & Missing Dynamics**
   - While the knowledge base successfully infers *which* motion achieves a state change, explain why a pure Description Logic or symbolic Prolog reasoner cannot verify whether a specific grasp force will cause a wet carton to slip, and identify which subsystem must handle those physical calculations.

---------------------

#### E7

The way I understand it is that `taskRequest` tells the executive what change would count as fulfilling the request.. so for ex. `open_fridge` means the fridge needs to become open. With `close_anything`, closing `drawer_04` or `drawer_05` can both satisfy the request.. each row is one possible answer.. at this point we have only specified what should change.. how the PR2 moves to make it happen comes in the next query



#### E8

Now the way I understand it is that `StateChange` connects the two parts of the query.. so for ex. `taskRequest` maps `open_fridge` to `open(fridge)`, and then `causes` uses that same `open(fridge)` to find the motion `envJointGoal` with the parameters `[0.8, b'fridge_door_joint', b'fridge_door_handle']`. So now the executive knows which handle to use and which joint to move.. and `0.8` is the target angle for the fridge door joint

The drawer example made this clearer for me.. `open(drawer_04)` gives the same motion type, BUT now the target is `0.2` for `drawer_04_joint` and the handle is `drawer_04_handle`. So the reasoner takes a request like “open this” and looks up a concrete motion goal.. the target tells us how far the door should rotate or the drawer should slide, and the joint and handle tell us exactly which object that applies to



#### E9

In E8 we found which motion could open the fridge.. now by adding `canPerform(Robot, Motion, MotionParam)` we’re also asking “which robot can actually carry out that motion with those parameters?”.. for ex. in the `open_fridge` result, `Robot` came back as `pr2`, meaning the knowledge base found that the PR2 can use the fridge handle to move the door joint to the `0.8` target

So `canPerform` acts as a filter here.. a motion only appears in the results if the reasoner can match it to a robot capable of performing it with those exact parameters. In my run the fridge and drawer results both matched `pr2`.. and in the RViz window I could see the robot slowly moving as it opened things in the simulation



#### E10

In my first run.. `open_anything` matched several targets.. for ex. both `open(drawer_04)` and `open(fridge)` showed up. When I replaced `Request` with `open_fridge` and ran the query again.. I got two identical rows and then `No more solutions`. Both rows had `open(fridge)` as the state change, with the same fridge door joint, handle and target value of `0.8`.. so there was only "one distinct goal" in the results

The search tree gets smaller because the drawer targets drop out at `taskRequest`.. the reasoner only needs to check which motion and robot can open the fridge. So even though the PR2 can open `drawer_04`, the executive can’t select “open drawer 04” as the goal for this request.. in this query I specifically asked the PR2 to open the fridge



#### E11

In my run the reasoner returned `envJointGoal` with `[0.8, b'fridge_door_joint', b'fridge_door_handle']`.. the handle tells the PR2 where to interact with the fridge, and `0.8` is the target angle in radians for the fridge door joint. So the `0.8` belongs to the "door".. the PR2 still has to work out how to move its own arm to open it

The motion system uses that goal to generate a sequence of joint positions over time.. the arm joints move together so the gripper can follow the handle as the door swings open. In the right window I could see the PR2 moving slowly through the opening motions.. `rviz -d .../bmp.rviz` opens the visualization that lets me watch those movements as the simulation updates



#### E12

The way I understand it is that `canPerform` can tell us the PR2 is capable of a motion.. BUT knowing it can do the lifting motion still leaves the question “will the wet carton actually stay in its grip?” Even if the knowledge base has the carton’s mass, grip force and friction coefficient, the symbolic rules need a physical law connecting those values before they can predict slipping

That calculation belongs to the "semantic digital twin’s causal model".. for an upward lift it checks $2\mu F \ge m(g+a)$, which basically asks whether the friction from both gripper contacts is enough to hold the carton as it accelerates upward. If the wet surface lowers the friction enough for that check to fail.. the model predicts the carton will slip

----------------------

# Submission guidelines

- Deadline: 24.11.2026, 4 PM
- Submission as Git repo commits pushed within deadlines (If not yet, create a repo with the name **iis-assignment-win26** for the whole semester and invite me as collaborator)
- Put the textual answers in notebook and all related media (images, rules, etc) should then be put into a folder called **Assignment3**. 